<a href="https://colab.research.google.com/github/sanadv/Classical_Machine_Learning/blob/main/Classical_Machine_Leaning_Chapter_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize image data to [0, 1] range
x_train = x_train / 255.0
x_test = x_test / 255.0

# One-hot encode the labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Resize CIFAR-10 images to match VGG16 input (224x224)
x_train = tf.image.resize(x_train, (224, 224))
x_test = tf.image.resize(x_test, (224, 224))

# Load pre-trained VGG16 model without the top layer (include_top=False)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze all convolutional layers in VGG16
for layer in base_model.layers:
    layer.trainable = False

# Build a new model on top of the pre-trained base
model = Sequential([
    base_model,
    Flatten(),  # Flatten feature maps
    Dense(256, activation='relu'),  # Add a dense layer
    Dense(10, activation='softmax')  # Final layer for 10 CIFAR-10 classes
])

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=10,
    batch_size=32
)
# Evaluate the model
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
# Plot training history
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()


In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize image data to [0, 1] range
x_train = x_train / 255.0
x_test = x_test / 255.0

# One-hot encode the labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Resize CIFAR-10 images to match DenseNet input (224x224)
x_train = tf.image.resize(x_train, (224, 224))
x_test = tf.image.resize(x_test, (224, 224))

# Load pre-trained DenseNet121 model without the top layer (include_top=False)
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Allow all layers to be trainable (fine-tuning)
for layer in base_model.layers:
    layer.trainable = True

# Build a new model on top of the pre-trained base
model = Sequential([
    base_model,
    Flatten(),  # Flatten feature maps
    Dropout(0.5),  # Add dropout for regularization
    Dense(256, activation='relu'),  # Add a dense layer
    Dense(10, activation='softmax')  # Final layer for 10 CIFAR-10 classes
])

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=10,
    batch_size=32
)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# Plot training history
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, GlobalAveragePooling2D
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

# Custom Cubixel Layer adjusted for feature maps
class Custom3DCubeLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(Custom3DCubeLayer, self).__init__(**kwargs)

    def call(self, inputs):
        adaptive_thresholds = self.calculate_adaptive_threshold(inputs)
        gap_images = tf.map_fn(lambda x: self.extract_features(x[0], x[1]),
                               (inputs, adaptive_thresholds), dtype=tf.float32)
        return gap_images

    @tf.function
    def extract_features(self, feature_map, threshold):
        # Apply operations on feature maps (not assuming RGB channels)
        cube_sizes = self.calculate_cube_sizes(feature_map)
        cube_properties = self.calculate_cube_properties(cube_sizes)
        similarities = self.compute_similarity(cube_properties, threshold)
        normalized_similarity = tf.math.sigmoid(similarities)
        edge_image = self.detect_edges(similarities, threshold)
        return edge_image

    @staticmethod
    def calculate_cube_sizes(feature_map):
        # Normalize feature map instead of assuming RGB values
        normalized_feature_map = tf.nn.sigmoid(feature_map)  # Normalize values between 0 and 1
        cube_sizes = normalized_feature_map * 2  # Example scaling
        return cube_sizes

    @staticmethod
    def calculate_cube_properties(cube_sizes):
        # Example feature map property extraction
        volume = tf.reduce_prod(cube_sizes, axis=-1)  # Compute volume across feature channels
        aspect_ratio = cube_sizes[..., 0] / (cube_sizes[..., 1] + 1e-6)  # Avoid division by zero
        surface_area = tf.reduce_sum(cube_sizes, axis=-1)  # Simplified surface area
        combined_properties = tf.stack([volume, aspect_ratio, surface_area], axis=-1)
        return combined_properties

    @staticmethod
    def compute_similarity(cube_properties, threshold):
        similarity = tf.zeros_like(cube_properties)
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                if dx == 0 and dy == 0:
                    continue
                shifted_properties = tf.roll(cube_properties, shift=[dx, dy], axis=[0, 1])
                similarity += tf.abs(cube_properties - shifted_properties)
        return similarity

    def detect_edges(self, similarity, threshold):
        levels = tf.where(
            similarity > threshold * 2, 3,
            tf.where(similarity > threshold, 2,
                     tf.where(similarity > threshold / 2, 1, 0)))
        return tf.cast(levels, tf.float32)

    @staticmethod
    def calculate_adaptive_threshold(feature_map):
        avg_size = tf.reduce_mean(feature_map, axis=[1, 2, 3])
        size_variability = tf.math.reduce_std(feature_map, axis=[1, 2, 3])
        return avg_size + size_variability

# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train / 255.0
x_test = x_test / 255.0
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Create a Dataset Pipeline with Resizing
batch_size = 32

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_dataset = train_dataset.map(lambda x, y: (tf.image.resize(x, (299, 299)), y),
                                  num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_dataset = test_dataset.map(lambda x, y: (tf.image.resize(x, (299, 299)), y),
                                num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Load pre-trained InceptionV3 model
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))

# Unfreeze all layers for fine-tuning
for layer in base_model.layers:
    layer.trainable = True

# Build model with Cubixel Layer
model = Sequential([
    base_model,
    Custom3DCubeLayer(),  # Integration of Custom Cubixel Layer
    GlobalAveragePooling2D(),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dense(10, activation='softmax')
])

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=10,
)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# Plot training history
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()
